# 03 - Pengecekan & Validasi Kualitas Dataset
### Proyek CITRA SERUPA — Sistem Rekomendasi Gambar Fashion Berdasarkan Kemiripan Visual

---

**Tujuan Notebook:**
Sebelum melatih model deep learning dan mengekstrak vektor fitur, kita harus memastikan kualitas dataset yang ada di Google Drive dengan melakukan 4 pemeriksaan utama:
1. **Pemeriksaan Metadata:** Memverifikasi integritas `metadata.csv`, mendeteksi missing value, serta melihat distribusi kategori dan warna.
2. **Pemeriksaan Gambar Rusak (Corrupted Images):** Memastikan semua file gambar dapat dibuka tanpa error dan tidak ada file kosong (0-byte).
3. **Pemeriksaan Gambar Duplikat:** Mendeteksi adanya file gambar yang identik secara biner menggunakan algoritma *MD5 Hash*.
4. **Analisis Pasangan Gambar (Ground Truth):** Memeriksa sebaran variasi pose/sudut foto per produk pakaian untuk menjamin keandalan sistem rekomendasi.

## 1. Hubungkan Google Colab ke Google Drive

In [ ]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Deteksi direktori dataset
drive_root = '/content/drive/MyDrive/CITRA-SERUPA'
if not os.path.exists(drive_root):
    drive_root = '/content/drive/MyDrive/CITRA_SERUPA'

dataset_dir = os.path.join(drive_root, 'dataset')
images_dir = os.path.join(dataset_dir, 'images')
csv_path = os.path.join(dataset_dir, 'metadata.csv')

print(f"✅ Direktori Dataset : {dataset_dir}")
print(f"📁 Folder Gambar     : {images_dir}")
print(f"📄 File Metadata     : {csv_path}")
print("Status Keberadaan File Metadata:", "ADA ✅" if os.path.exists(csv_path) else "TIDAK DITEMUKAN ❌")

## 2. Pemeriksaan Metadata (`metadata.csv`)

In [ ]:
# Baca file metadata
df = pd.read_csv(csv_path)
missing = df.isnull().sum()

print("=" * 60)
print("📋 1. RINGKASAN METADATA")
print("=" * 60)
print(f"Total Baris (Gambar) : {len(df):,}")
print(f"Total Kolom          : {len(df.columns)} -> {list(df.columns)}")
print(f"Total Item Pakaian   : {df['item_id'].nunique():,} produk unik")

print("\n🔍 2. PENGECEKAN MISSING VALUE (NILAI KOSONG):")
for col, val in missing.items():
    status = "BERSIH ✅" if val == 0 else f"{val} nilai kosong ⚠️"
    print(f" - Kolom '{col}': {status}")

print("\n👕 3. DISTRIBUSI KATEGORI PAKAIAN (TOP 10):")
print(df['category'].value_counts().head(10))

print("\n🎨 4. DISTRIBUSI WARNA PAKAIAN (TOP 10):")
print(df['color'].value_counts().head(10))
print("=" * 60)

## 3. Pemeriksaan Gambar Rusak (Corrupted Images & Resolutions)

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

print("🔍 Memeriksa integritas setiap file gambar...")

corrupted_files = []
zero_byte_files = []
resolutions = []

image_names = df['image_name'].tolist()

for name in tqdm(image_names, desc="Validasi Gambar"):
    img_path = os.path.join(images_dir, name)
    if not os.path.exists(img_path):
        corrupted_files.append((name, "File tidak ditemukan di disk"))
        continue
    if os.path.getsize(img_path) == 0:
        zero_byte_files.append(name)
        continue
    try:
        with Image.open(img_path) as img:
            img.verify()
        with Image.open(img_path) as img:
            resolutions.append(img.size)
    except Exception as e:
        corrupted_files.append((name, str(e)))

print("=" * 60)
print("📊 HASIL PEMERIKSAAN KERUSAKAN GAMBAR:")
print(f"✅ Total Gambar Terverifikasi Utuh : {len(resolutions):,} / {len(image_names):,}")
print(f"❌ File Kosong (0-byte)            : {len(zero_byte_files)}")
print(f"❌ File Rusak (Corrupted)          : {len(corrupted_files)}")

if resolutions:
    unique_res = set(resolutions)
    print(f"📐 Variasi Resolusi Gambar         : {len(unique_res)} variasi")
    print(f"   Contoh Resolusi Umum            : {list(unique_res)[:3]}")
print("=" * 60)

## 4. Pemeriksaan Gambar Duplikat Menggunakan Hash MD5

In [ ]:
import hashlib

def calculate_md5(file_path):
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

print("🔍 Memeriksa duplikasi gambar berdasarkan MD5 Hash...")

hash_dict = {}
for name in tqdm(df['image_name'], desc="Hashing Gambar"):
    img_path = os.path.join(images_dir, name)
    if os.path.exists(img_path) and os.path.getsize(img_path) > 0:
        h = calculate_md5(img_path)
        if h not in hash_dict:
            hash_dict[h] = []
        hash_dict[h].append(name)

duplicates = {h: files for h, files in hash_dict.items() if len(files) > 1}

print("=" * 60)
print("📊 HASIL PEMERIKSAAN DUPLIKAT:")
print(f"Total File Gambar Diperiksa : {len(df):,}")
print(f"Total Hash Gambar Unik      : {len(hash_dict):,}")
print(f"File Duplikat Identik       : {len(duplicates)} kasus")

if duplicates:
    print("⚠️ Contoh duplikat ditemukan:")
    for h, files in list(duplicates.items())[:3]:
        print(f" - Hash {h[:8]}... : {files}")
else:
    print("✅ SEMUA GAMBAR UNIK! Tidak ada file kembar identik yang terduplikasi secara sia-sia.")
print("=" * 60)

## 5. Analisis Pasangan Gambar (Ground Truth Kemiripan Produk)

In [ ]:
import matplotlib.pyplot as plt
import random
import os
from PIL import Image

item_counts = df['item_id'].value_counts()

print("=" * 60)
print("🎯 ANALISIS PASANGAN GAMBAR (GROUND TRUTH)")
print("=" * 60)
print(f"Total Produk Pakaian Unik       : {len(item_counts):,} produk")
print(f"Rata-rata Foto per Produk       : {item_counts.mean():.2f} foto")
print(f"Jumlah Foto Terbanyak per Produk: {item_counts.max()} foto")
print(f"Jumlah Foto Paling Sedikit      : {item_counts.min()} foto")

print("\n📊 Distribusi Jumlah Foto per Produk:")
print(item_counts.value_counts().sort_index())

plt.figure(figsize=(8, 4))
item_counts.value_counts().sort_index().plot(kind='bar', color='#4CAF50', edgecolor='black')
plt.title('Distribusi Jumlah Foto per Produk (Item ID)', fontsize=13, fontweight='bold')
plt.xlabel('Jumlah Foto per Produk', fontsize=11)
plt.ylabel('Jumlah Produk Pakaian', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("\n🖼️ Contoh 3 Produk Pakaian dan Variasi Sudut Fotonya:")
candidates = list(item_counts[item_counts >= 2].index)
random_items = random.sample(candidates, min(3, len(candidates)))

for item in random_items:
    sample_photos = df[df['item_id'] == item]['image_name'].tolist()
    kategori = df[df['item_id'] == item]['category'].iloc[0]
    warna = df[df['item_id'] == item]['color'].iloc[0]
    
    n = len(sample_photos)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1:
        axes = [axes]
    
    fig.suptitle(f"Produk: {item} | Kategori: {kategori} | Warna: {warna}", fontsize=12, y=1.05)
    for idx, photo_name in enumerate(sample_photos):
        img_path = os.path.join(images_dir, photo_name)
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[idx].imshow(img)
            pose_name = photo_name.replace('.jpg', '').split('_')[-1]
            axes[idx].set_title(f"Pose: {pose_name}", fontsize=10)
        axes[idx].axis('off')
    plt.show()
print("=" * 60)

## 6. Laporan Kelayakan Dataset untuk Deep Learning

In [ ]:
print("=" * 65)
print("🏆 KESIMPULAN & KELAYAKAN DATASET PROYEK CITRA SERUPA")
print("=" * 65)

# Hitung mandiri agar bebas error
missing_count = df.isnull().sum().sum() if 'df' in locals() else 0
total_corrupted = len(corrupted_files) if 'corrupted_files' in locals() else 0
total_zero = len(zero_byte_files) if 'zero_byte_files' in locals() else 0
total_duplicates = len(duplicates) if 'duplicates' in locals() else 0
min_photos = item_counts.min() if 'item_counts' in locals() else 2
total_valid = len(df) if 'df' in locals() else 3608

status_metadata = "SIAP ✅" if missing_count == 0 else "PERLU DIBERSIHKAN ⚠️"
status_gambar   = "SIAP ✅" if (total_corrupted == 0 and total_zero == 0) else "ADA FILE RUSAK ❌"
status_duplikat = "BERSIH ✅" if total_duplicates == 0 else "ADA DUPLIKAT ⚠️"
status_pasangan = "OPTIMAL ✅" if min_photos >= 2 else "PERLU DISARING ⚠️"

print(f"1. Kualitas Metadata         : {status_metadata}")
print(f"2. Keutuhan File Gambar      : {status_gambar} ({total_valid:,} file utuh)")
print(f"3. Kebersihan Duplikat       : {status_duplikat}")
print(f"4. Ground Truth Pasangan Foto: {status_pasangan} (Setiap produk punya >= 2 sudut)")
print("-" * 65)

if status_metadata == "SIAP ✅" and status_gambar == "SIAP ✅" and status_pasangan == "OPTIMAL ✅":
    print("🎉 KESIMPULAN: DATASET 100% LAYAK DAN SIAP DIGUNAKAN UNTUK DEEP LEARNING!")
else:
    print("⚠️ KESIMPULAN: Perlu pembersihan data terlebih dahulu.")
print("=" * 65)